## 课程三：韵律控制（45%）

实验目的：了解 VITS 模型的结构和工作原理，学习 MeloTTS 的韵律控制机制

MeloTTS 的模型架构和实现参考了 [VITS](https://github.com/jaywalnut310/vits), [VITS2](https://github.com/daniilrobnikov/vits2) and [Bert-VITS2](https://github.com/fishaudio/Bert-VITS2) 等开源项目。在本次课程中，你需要了解 VITS 模型的结构和工作原理，并尝试控制 MeloTTS 的韵律。

请阅读和观看以下资料：
- [VITS 论文](https://arxiv.org/abs/2106.06103)
- [VITS 讲解视频](https://www.bilibili.com/video/BV1Jb4y1g7q6/?p=3&share_source=copy_web&vd_source=d0dab92918e0499fec84fd963e2a879e)

![vits](./vits.png)

> 【问题一】 请介绍一下 VITS 模型的组成结构，训练和推理过程的工作原理。

> 【问题二】 请介绍一下 VITS 中的 Monotonic Alignment Search (MAS) 算法和 Stochastic Duration Predictor 模块，说明它们在时长建模和韵律生成中的作用。

MeloTTS 使用 Stochastic Duration Predictor 模块来实现韵律控制机制的代码在 `melo/models.py` 的 `infer` 函数中（966 行），其中一个重要的输出是 `w_ceil`，它表示了每个音素的持续时间。可以通过自定义的 `get_original_w_ceil` 函数来获取 `w_ceil`：

In [2]:
from melo.api import TTS
import pandas as pd

pd.set_option('display.max_columns', None)

# Speed is adjustable
speed = 1
device = 'cuda' # or cuda:0

text = "落霞与孤鹜齐飞，秋水共长天一色。"
model = TTS(language='ZH', device=device)
speaker_ids = model.hps.data.spk2id

output_path = 'zh.wav'

# 获取原始的 phones、tones、w_ceil 列表
w_ceil_list, phone_list, tone_list = model.get_original_w_ceil(text, speaker_ids['ZH'], output_path, speed=speed, sdp_ratio=0, noise_scale=0, noise_scale_w=0)

# 将 phones 列表中的 ID 转换为对应的符号
symbol_to_id_map = model.symbol_to_id
id_to_symbol_map = {v: k for k, v in zip(symbol_to_id_map.keys(), symbol_to_id_map.values())}
print(f'id_to_symbol_map: {id_to_symbol_map}')
# for w_ceil, phones, tones in zip(w_ceil_list, phone_list, tone_list):
#     print('phones:', phones.flatten().tolist())
#     print('tones:', tones.flatten().tolist())
#     print('w_ceil:', w_ceil.flatten().int().tolist())

print(f'w_ceil_list: {w_ceil_list}')

df = pd.DataFrame({
    'phones': [id_to_symbol_map.get(item, '') for sublist in phone_list for item in sublist.flatten().tolist()],
    'tones': [item for sublist in tone_list for item in sublist.flatten().tolist()],
    'w_ceil': [item for sublist in w_ceil_list for item in sublist.flatten().int().tolist()]
})


df.T

d:\miniconda3\envs\env_speech\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


 > Text split to sentences.
落霞与孤鹜齐飞, 秋水共长天一色.
 > ===========================


  0%|          | 0/1 [00:00<?, ?it/s]Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\Guan\AppData\Local\Temp\jieba.cache
Loading model cost 0.581 seconds.
Prefix dict has been built successfully.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-multilingual-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 1/1 [00:16<00:00, 16.18s/it]

id_to_symbol_map: {0: '_', 1: 'AA', 2: 'E', 3: 'EE', 4: 'En', 5: 'N', 6: 'OO', 7: 'V', 8: 'a', 9: 'a:', 10: 'aa', 11: 'ae', 12: 'ah', 13: 'ai', 14: 'an', 15: 'ang', 16: 'ao', 17: 'aw', 18: 'ay', 19: 'b', 20: 'by', 21: 'c', 22: 'ch', 23: 'd', 24: 'dh', 25: 'dy', 26: 'e', 27: 'e:', 28: 'eh', 29: 'ei', 30: 'en', 31: 'eng', 32: 'er', 33: 'ey', 34: 'f', 35: 'g', 36: 'gy', 37: 'h', 38: 'hh', 39: 'hy', 40: 'i', 41: 'i0', 42: 'i:', 43: 'ia', 44: 'ian', 45: 'iang', 46: 'iao', 47: 'ie', 48: 'ih', 49: 'in', 50: 'ing', 51: 'iong', 52: 'ir', 53: 'iu', 54: 'iy', 55: 'j', 56: 'jh', 57: 'k', 58: 'ky', 59: 'l', 60: 'm', 61: 'my', 62: 'n', 63: 'ng', 64: 'ny', 65: 'o', 66: 'o:', 67: 'ong', 68: 'ou', 69: 'ow', 70: 'oy', 71: 'p', 72: 'py', 73: 'q', 74: 'r', 75: 'ry', 76: 's', 77: 'sh', 78: 't', 79: 'th', 80: 'ts', 81: 'ty', 82: 'u', 83: 'u:', 84: 'ua', 85: 'uai', 86: 'uan', 87: 'uang', 88: 'uh', 89: 'ui', 90: 'un', 91: 'uo', 92: 'uw', 93: 'v', 94: 'van', 95: 've', 96: 'vn', 97: 'w', 98: 'x', 99: 'y', 100: 

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64
phones,_,_,_,l,_,uo,_,x,_,ia,_,y,_,v,_,g,_,u,_,w,_,u,_,q,_,i,_,f,_,ei,_,",",_,q,_,iu,_,sh,_,ui,_,g,_,ong,_,ch,_,ang,_,t,_,ian,_,y,_,i,_,s,_,e,_,.,_,_,_
tones,0,0,0,4,0,4,0,2,0,2,0,3,0,3,0,1,0,1,0,4,0,4,0,2,0,2,0,1,0,1,0,0,0,1,0,1,0,3,0,3,0,4,0,4,0,2,0,2,0,1,0,1,0,2,0,2,0,4,0,4,0,0,0,0,0
w_ceil,7,7,15,2,3,3,2,6,4,5,3,4,4,3,6,3,3,3,3,3,3,3,3,11,4,3,3,4,6,6,4,4,7,5,2,5,3,8,4,4,4,3,4,4,3,4,4,6,4,5,3,3,3,3,4,3,5,6,5,5,4,3,3,3,4


从以上 pandas DataFrame 中可以看到每个音素对应的持续时间（w_ceil），w_ceil 的一个单位对应 512 (hop_length) / 44100 (采样率) ≈ 0.0116s = 11.6ms 的时间长度。因此，可以通过调整 `w_ceil_list` 中每个 w_ceil 的值，来控制每个音素的持续时间，从而实现简单的韵律控制。

In [3]:
# w_ceil_list = [[7, 7, 15, 2, 3, 3, 2, 6, 4, 5, 3, 4, 4, 3, 6, 3, 9, 9, 9, 9, 9, 9, 9, 11, 4, 3, 3, 4, 6, 6, 4, 4, 7, 5, 2, 5, 3, 8, 4, 4, 4, 3, 4, 4, 3, 8, 8, 18, 12, 15, 12, 12, 12, 3, 4, 3, 5, 6, 5, 5, 4, 3, 3, 3, 4]]
# w_ceil_list = [[7, 7, 15, 2, 3, 3, 2, 6, 4, 5, 3, 4, 4, 3, 6, 3, 9, 9, 9, 9, 9, 9, 9, 11, 4, 3, 3, 4, 6, 6, 4, 4, 7, 5, 2, 5, 3, 8, 4, 4, 4, 3, 4, 4, 3, 8, 8, 18, 12, 15, 12, 12, 12, 3, 4, 3, 5, 6, 5, 5, 4, 3, 3, 3, 4]]
modified_w_ceil_list = w_ceil_list[0].squeeze(0).int().tolist()
modified_w_ceil_list[0][15:23] = [9] * 8 # 延长 “孤” 和 “鹜” 音素的持续时间为 9 个单位（约 104ms）
modified_w_ceil_list[0][45:53] = [9] * 8 # 延长 “长” 和 “天” 音素的持续时间为 9 个单位（约 104ms）

# 修改后的 w_ceil_list
print(f'modified w_ceil_list: {modified_w_ceil_list}')


modified w_ceil_list: [[7, 7, 15, 2, 3, 3, 2, 6, 4, 5, 3, 4, 4, 3, 6, 9, 9, 9, 9, 9, 9, 9, 9, 11, 4, 3, 3, 4, 6, 6, 4, 4, 7, 5, 2, 5, 3, 8, 4, 4, 4, 3, 4, 4, 3, 9, 9, 9, 9, 9, 9, 9, 9, 3, 4, 3, 5, 6, 5, 5, 4, 3, 3, 3, 4]]


In [4]:
# 按照原始的韵律生成语音，关闭随机性以便对比
model.tts_to_file(text, speaker_ids['ZH'], output_path, speed=speed, sdp_ratio=0, noise_scale=0, noise_scale_w=0)

from IPython.display import Audio

Audio(output_path)

 > Text split to sentences.
落霞与孤鹜齐飞, 秋水共长天一色.
 > ===========================


100%|██████████| 1/1 [00:00<00:00,  2.01it/s]


In [5]:
# 按照调整后的韵律生成语音，使用自定义的 w_ceil_list 来控制每个音素的持续时间，关闭随机性以便对比
model.tts_to_file_custom_duration(text, speaker_ids['ZH'], output_path, speed=speed, sdp_ratio=0, noise_scale=0, noise_scale_w=0, w_ceil_customized=modified_w_ceil_list)

from IPython.display import Audio

Audio(output_path)

 > Text split to sentences.
落霞与孤鹜齐飞, 秋水共长天一色.
 > ===========================


100%|██████████| 1/1 [00:00<00:00,  3.78it/s]


可以观察到修改后的音频中，"孤"和"鹜"这两个字的音素的持续时间被延长了，"长"和"天"这两个字的音素的持续时间也被延长了，从而产生更突出的节奏和停连效果。

> 【问题三】 请通过调整 `w_ceil` 来控制合成语音的韵律，合成一句 “你听到了吗？这句话我会说得越来越慢。” 的语音。要求语速越来越慢：句首接近原始时长、句尾约为原始时长的 3 倍（对应语速约从 1 倍`线性`降至 0.33 倍）。提交代码和生成的音频文件（命名为 `homework_3.wav`）。

In [6]:
from melo.api import TTS
import pandas as pd

pd.set_option('display.max_columns', None)

# Speed is adjustable
speed = 1
device = 'cuda' # or cuda:0

text = "你听到了吗？这句话我会说得越来越慢。"
model = TTS(language='ZH', device=device)
speaker_ids = model.hps.data.spk2id

output_path = 'homework_3.wav'

# 获取原始的 phones、tones、w_ceil 列表
w_ceil_list, phone_list, tone_list = model.get_original_w_ceil(text, speaker_ids['ZH'], output_path, speed=speed, sdp_ratio=0, noise_scale=0, noise_scale_w=0)

# 将 phones 列表中的 ID 转换为对应的符号
symbol_to_id_map = model.symbol_to_id
id_to_symbol_map = {v: k for k, v in zip(symbol_to_id_map.keys(), symbol_to_id_map.values())}
print(f'id_to_symbol_map: {id_to_symbol_map}')
# for w_ceil, phones, tones in zip(w_ceil_list, phone_list, tone_list):
#     print('phones:', phones.flatten().tolist())
#     print('tones:', tones.flatten().tolist())
#     print('w_ceil:', w_ceil.flatten().int().tolist())

print(f'w_ceil_list: {w_ceil_list}')

df = pd.DataFrame({
    'phones': [id_to_symbol_map.get(item, '') for sublist in phone_list for item in sublist.flatten().tolist()],
    'tones': [item for sublist in tone_list for item in sublist.flatten().tolist()],
    'w_ceil': [item for sublist in w_ceil_list for item in sublist.flatten().int().tolist()]
})


df.T

d:\miniconda3\envs\env_speech\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


 > Text split to sentences.
你听到了吗. 这句话我会说得越来越慢.
 > ===========================


100%|██████████| 1/1 [00:00<00:00, 12.45it/s]

id_to_symbol_map: {0: '_', 1: 'AA', 2: 'E', 3: 'EE', 4: 'En', 5: 'N', 6: 'OO', 7: 'V', 8: 'a', 9: 'a:', 10: 'aa', 11: 'ae', 12: 'ah', 13: 'ai', 14: 'an', 15: 'ang', 16: 'ao', 17: 'aw', 18: 'ay', 19: 'b', 20: 'by', 21: 'c', 22: 'ch', 23: 'd', 24: 'dh', 25: 'dy', 26: 'e', 27: 'e:', 28: 'eh', 29: 'ei', 30: 'en', 31: 'eng', 32: 'er', 33: 'ey', 34: 'f', 35: 'g', 36: 'gy', 37: 'h', 38: 'hh', 39: 'hy', 40: 'i', 41: 'i0', 42: 'i:', 43: 'ia', 44: 'ian', 45: 'iang', 46: 'iao', 47: 'ie', 48: 'ih', 49: 'in', 50: 'ing', 51: 'iong', 52: 'ir', 53: 'iu', 54: 'iy', 55: 'j', 56: 'jh', 57: 'k', 58: 'ky', 59: 'l', 60: 'm', 61: 'my', 62: 'n', 63: 'ng', 64: 'ny', 65: 'o', 66: 'o:', 67: 'ong', 68: 'ou', 69: 'ow', 70: 'oy', 71: 'p', 72: 'py', 73: 'q', 74: 'r', 75: 'ry', 76: 's', 77: 'sh', 78: 't', 79: 'th', 80: 'ts', 81: 'ty', 82: 'u', 83: 'u:', 84: 'ua', 85: 'uai', 86: 'uan', 87: 'uang', 88: 'uh', 89: 'ui', 90: 'un', 91: 'uo', 92: 'uw', 93: 'v', 94: 'van', 95: 've', 96: 'vn', 97: 'w', 98: 'x', 99: 'y', 100: 

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72
phones,_,_,_,n,_,i,_,t,_,ing,_,d,_,ao,_,l,_,e,_,m,_,a,_,.,_,zh,_,e,_,j,_,v,_,h,_,ua,_,w,_,o,_,h,_,ui,_,sh,_,uo,_,d,_,e,_,y,_,ve,_,l,_,ai,_,y,_,ve,_,m,_,an,_,.,_,_,_
tones,0,0,0,3,0,3,0,1,0,1,0,4,0,4,0,5,0,5,0,5,0,5,0,0,0,4,0,4,0,4,0,4,0,4,0,4,0,3,0,3,0,4,0,4,0,1,0,1,0,5,0,5,0,4,0,4,0,2,0,2,0,4,0,4,0,4,0,4,0,0,0,0,0
w_ceil,7,10,13,2,3,3,3,7,3,2,4,3,3,3,3,2,2,2,5,5,5,3,5,3,8,4,3,2,2,3,3,3,2,4,3,4,3,2,3,2,2,3,4,3,4,4,3,3,2,2,3,3,5,3,6,5,3,3,3,3,3,3,3,3,7,3,5,4,4,3,3,3,4


In [10]:
modified_w_ceil_list = w_ceil_list[0].squeeze(0).int().tolist()
speed_increment = 2 / len(modified_w_ceil_list[0])  # 计算每个音素的持续时间增加量
speed_factor = 1.0  # 初始速度因子

for i in range(len(modified_w_ceil_list[0])):
    modified_w_ceil_list[0][i] = int(modified_w_ceil_list[0][i] * speed_factor)
    speed_factor += speed_increment  # 增加速度因子

print(f"modified_w_ceil_list: {modified_w_ceil_list}")

modified_w_ceil_list: [[7, 10, 13, 2, 3, 3, 3, 8, 3, 2, 5, 3, 3, 4, 4, 2, 2, 2, 7, 7, 7, 4, 8, 4, 13, 6, 5, 3, 3, 5, 5, 5, 3, 7, 5, 7, 5, 4, 6, 4, 4, 6, 8, 6, 8, 8, 6, 6, 4, 4, 7, 7, 12, 7, 14, 12, 7, 7, 7, 7, 7, 8, 8, 8, 19, 8, 14, 11, 11, 8, 8, 8, 11]]


In [11]:
# 原始速率
model.tts_to_file(text, speaker_ids['ZH'], output_path, speed=speed, sdp_ratio=0, noise_scale=0, noise_scale_w=0)

from IPython.display import Audio

Audio(output_path)

 > Text split to sentences.
你听到了吗. 这句话我会说得越来越慢.
 > ===========================


100%|██████████| 1/1 [00:00<00:00,  4.06it/s]


In [12]:
# 调整后速率
model.tts_to_file_custom_duration(text, speaker_ids['ZH'], output_path, speed=speed, sdp_ratio=0, noise_scale=0, noise_scale_w=0, w_ceil_customized=modified_w_ceil_list)

from IPython.display import Audio

Audio(output_path)

 > Text split to sentences.
你听到了吗. 这句话我会说得越来越慢.
 > ===========================


100%|██████████| 1/1 [00:00<00:00,  2.05it/s]
